# Neon Web Traffic — Sample Logs

Dashboard exported from example-mcp-dashbuilder


In [ ]:
from elasticsearch import Elasticsearch
import pandas as pd
import matplotlib.pyplot as plt

# Update with your Elasticsearch credentials
es = Elasticsearch(
    "http://localhost:9200",
    # api_key="your-api-key-here"
    basic_auth=("elastic", "changeme")
)


/Users/stratoula/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Request lines

Chart type: **metric**


In [2]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS lines = COUNT(*)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
print(f"Request lines: {df.iloc[0]['lines']}")
plt.tight_layout()
plt.show()


AuthenticationException: AuthenticationException(401, 'security_exception', 'missing authentication credentials for REST request [/_query?format=json]')

In [ ]:
# Request lines — trend sparkline
trend_result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS lines = COUNT(*) BY ts = BUCKET(@timestamp, 1 day) | SORT ts
    """,
    format="json"
)

trend_df = pd.DataFrame(trend_result["values"], columns=[c["name"] for c in trend_result["columns"]])
trend_df.plot(title="Request lines — Trend")
plt.tight_layout()
plt.show()


## Data shipped

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS mib = ROUND(SUM(bytes)/1048576, 1)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
print(f"Data shipped: {df.iloc[0]['mib']}")
plt.tight_layout()
plt.show()


## Geo destinations

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS regions = COUNT_DISTINCT(geo.dest)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
print(f"Geo destinations: {df.iloc[0]['regions']}")
plt.tight_layout()
plt.show()


## Error share

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS total = COUNT(*), errors = COUNT(*) WHERE TO_INTEGER(`response.keyword`) >= 400 | EVAL err_pct = ROUND(errors * 100.0 / total, 2)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
print(f"Error share: {df.iloc[0]['err_pct']}")
plt.tight_layout()
plt.show()


## Traffic aurora (hourly)

Chart type: **area**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS requests = COUNT(*) BY ts = BUCKET(@timestamp, 1 hour) | SORT ts
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.plot.area(x="ts", y=["requests"], title="Traffic aurora (hourly)")
plt.tight_layout()
plt.show()


## HTTP response codes

Chart type: **bar**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS hits = COUNT(*) BY code = response.keyword | SORT hits DESC | LIMIT 10
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.plot.bar(x="code", y=["hits"], title="HTTP response codes")
plt.tight_layout()
plt.show()


## OS mix (top 6)

Chart type: **pie**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS hits = COUNT(*) BY os = machine.os.keyword | SORT hits DESC | LIMIT 6
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.set_index("os")[["hits"]].plot.pie(subplots=True, title="OS mix (top 6)")
plt.tight_layout()
plt.show()


## When the world pings us (day × hour)

Chart type: **heatmap**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | EVAL day = DATE_FORMAT("EEEE", @timestamp), hour = DATE_FORMAT("HH", @timestamp) | STATS hits = COUNT(*) BY day, hour | SORT day, hour
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.pivot_table(index="day", columns="hour", values="hits").style.background_gradient()
plt.tight_layout()
plt.show()
